# 03 · Experiments & tuning

Two tools for getting a number you can put in a paper:

- `Experiment` — run *datasets × models × metrics × runs* and report mean ± std.
- `RandomSearchExperiment` — sample hyper-parameters and keep the best.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # quiet TensorFlow

import numpy as np
import scipy.sparse as sp


def make_synthetic(n_per=60, k=3, dims=(30, 20), sep=1.0,
                   noise=(1.0, 2.5), p_noisy_edge=0.3, seed=0):
    """A small multi-view dataset with an uneven pair of views: (Xs, As, labels).

    Each view is Gaussian blobs around k centres. View 1 is deliberately
    noisier than view 0, and each graph mixes intra-cluster edges with a
    fraction of random ones -- so view weighting and propagation both have
    something real to do.
    """
    rng = np.random.RandomState(seed)
    n = n_per * k
    labels = np.repeat(np.arange(k), n_per)
    Xs, As = [], []
    for d, sigma in zip(dims, noise):
        centres = rng.randn(k, d) * sep
        Xs.append((centres[labels] + rng.randn(n, d) * sigma).astype(np.float64))
        A = np.zeros((n, n))
        for i in range(n):
            same = np.where(labels == labels[i])[0]
            A[i, rng.choice(same, size=5, replace=False)] = 1.0
            if rng.rand() < p_noisy_edge:
                A[i, rng.randint(n)] = 1.0          # a wrong edge
        As.append(sp.csr_matrix(np.maximum(A, A.T)))
    return Xs, As, labels


Xs, As, labels = make_synthetic()

# --- To use a real benchmark instead (downloads + caches on first call): ---
# from polyclustx.data import load_dataset
# data = load_dataset("acm")
# Xs, As, labels = data.Xs, data.As, data.labels

k = len(np.unique(labels))
print(f"views={len(Xs)}  samples={Xs[0].shape[0]}  clusters={k}")
print("view feature dims:", [X.shape[1] for X in Xs])

## Repeated comparison

`runs > 1` re-fits each model from scratch every time. Because LMGEC's
initialisation is randomized, the spread across runs is the honest error bar.

In [ ]:
from polyclustx.experiment import Experiment
from polyclustx.models import LMGEC
from polyclustx.metrics import clustering_accuracy, nmi_score, ari_score

result = Experiment(
    datasets={"synthetic": (Xs, As, labels)},
    models=[
        LMGEC(n_clusters=k, embedding_dim=8),
        LMGEC(n_clusters=k, embedding_dim=16),
        LMGEC(n_clusters=k, embedding_dim=32),
    ],
    metrics=[clustering_accuracy, nmi_score, ari_score],
    runs=5,
).run()

Results are also available programmatically, so you can plot or export them.

In [ ]:
for dataset_name, rows in result.all_results.items():
    for r in rows:
        acc = r.metric_avg["clustering_accuracy"]
        std = r.metric_std["clustering_accuracy"]
        print(f"{dataset_name:>10s} · {r.model_name:<24s} ACC = {acc:.3f} ± {std:.3f}")

## Random search

Give a distribution per parameter — a list is sampled uniformly, a
`(low, high)` tuple is sampled from that range — and the metric to optimise.

In [ ]:
from polyclustx.experiment import RandomSearchExperiment

search = RandomSearchExperiment(
    model_class=LMGEC,
    param_distributions={
        "n_clusters": [k],
        "embedding_dim": [8, 16, 32],
        "temperature": [0.1, 1.0, 5.0, 10.0],
        "beta": [0.5, 1.0, 2.0],
    },
    dataset=(Xs, As, labels),
    metrics=[clustering_accuracy, nmi_score],
    best_metric="clustering_accuracy",
    n_iter=10,
    seed=0,
).run()

print("best params:", search.best_params)
print("best score :", round(search.best_score, 4))

## Refit the winner

The best configuration is just a dict of keyword arguments.

In [ ]:
best = LMGEC(**search.best_params).fit(Xs, As)
print(f"ACC = {clustering_accuracy(labels, best.labels_):.3f}")
print(f"NMI = {nmi_score(labels, best.labels_):.3f}")

## The same thing from the shell

Everything above is also a CLI command, which is usually what you want on a
cluster:

```bash
polyclustx-cli --dataset acm --embedding-dim 16 --runs 5
polyclustx-cli --dataset acm --random-search --n-trials 20 --best-metric acc
polyclustx-cli --config experiment          # a bundled config, by shorthand name
```